In [1]:
import pandas as pd

df = pd.read_csv("C:/Users/Admin/Downloads/city_day.csv")
df.head()



,City,Date,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,O3,Benzene,Toluene,Xylene,AQI,AQI_Bucket
0,Ahmedabad,2015-01-01,NaN,NaN,0.92,18.22,17.15,NaN,0.92,27.64,133.36,0.00,0.02,0.00,NaN,NaN
1,Ahmedabad,2015-01-02,NaN,NaN,0.97,15.69,16.46,NaN,0.97,24.55,34.06,3.68,5.50,3.77,NaN,NaN
2,Ahmedabad,2015-01-03,NaN,NaN,17.40,19.30,29.70,NaN,17.40,29.07,30.70,6.80,16.40,2.25,NaN,NaN
3,Ahmedabad,2015-01-04,NaN,NaN,1.70,18.48,17.97,NaN,1.70,18.59,36.08,4.43,10.14,1.00,NaN,NaN
4,Ahmedabad,2015-01-05,NaN,NaN,22.10,21.42,37.76,NaN,22.10,39.33,39.31,7.01,18.89,2.78,NaN,NaN


In [5]:
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Drop AQI_Bucket (non-numeric column)
df = df.drop('AQI_Bucket', axis=1)

# Handle missing values using mean
imputer = SimpleImputer(strategy='mean')
df_imputed = pd.DataFrame(imputer.fit_transform(df), columns=df.columns)

# Create AQI categories for classification (0 = Good, 4 = Hazardous)
df_imputed['AQI_Category'] = pd.cut(df_imputed['AQI'],
                                    bins=[0, 100, 200, 300, 400, 500],
                                    labels=[0, 1, 2, 3, 4])
df_imputed.dropna(subset=['AQI_Category'], inplace=True)
df_imputed['AQI_Category'] = df_imputed['AQI_Category'].astype(int)

# Split into features and labels
X = df_imputed.drop(['AQI', 'AQI_Category'], axis=1)
y = df_imputed['AQI_Category']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [6]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

svm_model = SVC()
svm_model.fit(X_train_scaled, y_train)

svm_preds = svm_model.predict(X_test_scaled)

print("🎯 SVM Accuracy:", accuracy_score(y_test, svm_preds))
print("\n📊 Confusion Matrix:\n", confusion_matrix(y_test, svm_preds))
print("\n📄 Classification Report:\n", classification_report(y_test, svm_preds))


🎯 SVM Accuracy: 0.8094170403587444

📊 Confusion Matrix:
 [[1729  256    0    0    0]
 [ 344 2216   72   14    4]
 [   4  186  288   61    2]
 [   1    8   65  360   18]
 [   0    5    3   62  100]]

📄 Classification Report:
               precision    recall  f1-score   support

           0       0.83      0.87      0.85      1985
           1       0.83      0.84      0.83      2650
           2       0.67      0.53      0.59       541
           3       0.72      0.80      0.76       452
           4       0.81      0.59      0.68       170

    accuracy                           0.81      5798
   macro avg       0.77      0.72      0.74      5798
weighted avg       0.81      0.81      0.81      5798



In [7]:
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Initialize and train the model
nb_model = GaussianNB()
nb_model.fit(X_train_scaled, y_train)

# Make predictions
nb_preds = nb_model.predict(X_test_scaled)

# Evaluate
print("🎯 Naive Bayes Accuracy:", accuracy_score(y_test, nb_preds))
print("\n📊 Confusion Matrix:\n", confusion_matrix(y_test, nb_preds))
print("\n📄 Classification Report:\n", classification_report(y_test, nb_preds))


🎯 Naive Bayes Accuracy: 0.7149016902380131

📊 Confusion Matrix:
 [[1695  263   19    6    2]
 [ 478 1960  135   38   39]
 [   8  306  170   43   14]
 [   0   85   78  226   63]
 [   0    3    5   68   94]]

📄 Classification Report:
               precision    recall  f1-score   support

           0       0.78      0.85      0.81      1985
           1       0.75      0.74      0.74      2650
           2       0.42      0.31      0.36       541
           3       0.59      0.50      0.54       452
           4       0.44      0.55      0.49       170

    accuracy                           0.71      5798
   macro avg       0.60      0.59      0.59      5798
weighted avg       0.71      0.71      0.71      5798



In [8]:
from sklearn.tree import DecisionTreeClassifier

# Initialize and train the model
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train_scaled, y_train)

# Make predictions
dt_preds = dt_model.predict(X_test_scaled)

# Evaluate
print("🎯 Decision Tree Accuracy:", accuracy_score(y_test, dt_preds))
print("\n📊 Confusion Matrix:\n", confusion_matrix(y_test, dt_preds))
print("\n📄 Classification Report:\n", classification_report(y_test, dt_preds))


🎯 Decision Tree Accuracy: 0.7669886167644016

📊 Confusion Matrix:
 [[1606  369    9    1    0]
 [ 348 2140  143   15    4]
 [  10  151  293   82    5]
 [   1   24   89  301   37]
 [   0    7    2   54  107]]

📄 Classification Report:
               precision    recall  f1-score   support

           0       0.82      0.81      0.81      1985
           1       0.80      0.81      0.80      2650
           2       0.55      0.54      0.54       541
           3       0.66      0.67      0.67       452
           4       0.70      0.63      0.66       170

    accuracy                           0.77      5798
   macro avg       0.70      0.69      0.70      5798
weighted avg       0.77      0.77      0.77      5798



In [9]:
from sklearn.ensemble import RandomForestClassifier

# Initialize and train the model
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_scaled, y_train)

# Make predictions
rf_preds = rf_model.predict(X_test_scaled)

# Evaluate
print("🎯 Random Forest Accuracy:", accuracy_score(y_test, rf_preds))
print("\n📊 Confusion Matrix:\n", confusion_matrix(y_test, rf_preds))
print("\n📄 Classification Report:\n", classification_report(y_test, rf_preds))


🎯 Random Forest Accuracy: 0.8311486719558469

📊 Confusion Matrix:
 [[1743  241    1    0    0]
 [ 251 2296   85   12    6]
 [   1  159  318   63    0]
 [   0   15   59  363   15]
 [   0    4    4   63   99]]

📄 Classification Report:
               precision    recall  f1-score   support

           0       0.87      0.88      0.88      1985
           1       0.85      0.87      0.86      2650
           2       0.68      0.59      0.63       541
           3       0.72      0.80      0.76       452
           4       0.82      0.58      0.68       170

    accuracy                           0.83      5798
   macro avg       0.79      0.74      0.76      5798
weighted avg       0.83      0.83      0.83      5798



In [10]:
from sklearn.neighbors import KNeighborsClassifier

# Initialize the model (k=5 is a good starting point)
knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train_scaled, y_train)

# Make predictions
knn_preds = knn_model.predict(X_test_scaled)

# Evaluate
print("🎯 KNN Accuracy:", accuracy_score(y_test, knn_preds))
print("\n📊 Confusion Matrix:\n", confusion_matrix(y_test, knn_preds))
print("\n📄 Classification Report:\n", classification_report(y_test, knn_preds))


🎯 KNN Accuracy: 0.8025181096929976

📊 Confusion Matrix:
 [[1743  241    1    0    0]
 [ 331 2223   79   13    4]
 [   6  199  271   63    2]
 [   1   18   91  319   23]
 [   0    2    4   67   97]]

📄 Classification Report:
               precision    recall  f1-score   support

           0       0.84      0.88      0.86      1985
           1       0.83      0.84      0.83      2650
           2       0.61      0.50      0.55       541
           3       0.69      0.71      0.70       452
           4       0.77      0.57      0.66       170

    accuracy                           0.80      5798
   macro avg       0.75      0.70      0.72      5798
weighted avg       0.80      0.80      0.80      5798



In [12]:
!pip install lightgbm
import lightgbm as lgb

# Initialize and train the model
lgb_model = lgb.LGBMClassifier(random_state=42)
lgb_model.fit(X_train_scaled, y_train)

# Make predictions
lgb_preds = lgb_model.predict(X_test_scaled)

# Evaluate
print("🎯 LightGBM Accuracy:", accuracy_score(y_test, lgb_preds))
print("\n📊 Confusion Matrix:\n", confusion_matrix(y_test, lgb_preds))
print("\n📄 Classification Report:\n", classification_report(y_test, lgb_preds))



[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ------- -------------------------------- 0.3/1.5 MB ? eta -:--:--
   ---------------------------- ----------- 1.0/1.5 MB 3.1 MB/s eta 0:00:01
   ------------------------------------ --- 1.3/1.5 MB 3.4 MB/s eta 0:00:01
   ---------------------------------------- 1.5/1.5 MB 2.1 MB/s eta 0:00:00
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002118 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3060
[LightGBM] [Info] Number of data points in the train set: 23190, number of used features: 12
[LightGBM] [Info] Start training from score -1.118208
[LightGBM] [Info] Start training from score -0.758635
[LightGBM] [Info] Start training from score -2.337245
[LightGBM] [Info] Start training from score -2.509793
[LightGBM] [Info] Start training from score -3.613725
🎯 LightGBM Accuracy: 0.8345981372887202

📊 Confusion Matrix:
 [[1766  218  

C:\python 3.9\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [13]:
!pip install xgboost
from xgboost import XGBClassifier

# Initialize and train the model
xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='mlogloss', random_state=42)
xgb_model.fit(X_train_scaled, y_train)

# Make predictions
xgb_preds = xgb_model.predict(X_test_scaled)

# Evaluate
print("🎯 XGBoost Accuracy:", accuracy_score(y_test, xgb_preds))
print("\n📊 Confusion Matrix:\n", confusion_matrix(y_test, xgb_preds))
print("\n📄 Classification Report:\n", classification_report(y_test, xgb_preds))


   ---------------------------------------- 0.0/124.9 MB ? eta -:--:--
   ---------------------------------------- 0.3/124.9 MB ? eta -:--:--
   ---------------------------------------- 1.3/124.9 MB 3.7 MB/s eta 0:00:34
    --------------------------------------- 2.1/124.9 MB 4.1 MB/s eta 0:00:31
   - -------------------------------------- 3.1/124.9 MB 4.1 MB/s eta 0:00:30
   - -------------------------------------- 4.2/124.9 MB 4.4 MB/s eta 0:00:28
   - -------------------------------------- 5.5/124.9 MB 4.5 MB/s eta 0:00:27
   -- ------------------------------------- 6.6/124.9 MB 4.7 MB/s eta 0:00:26
   -- ------------------------------------- 7.9/124.9 MB 4.9 MB/s eta 0:00:25
   -- ------------------------------------- 8.9/124.9 MB 4.9 MB/s eta 0:00:24
   --- ------------------------------------ 10.0/124.9 MB 4.9 MB/s eta 0:00:24
   --- ------------------------------------ 11.0/124.9 MB 4.8 MB/s eta 0:00:24
   --- ------------------------------------ 12.3/124.9 MB 4.9 MB/s eta 0:00:


[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip
C:\python 3.9\lib\site-packages\xgboost\core.py:158: UserWarning: [22:04:10] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


🎯 XGBoost Accuracy: 0.8308037254225595

📊 Confusion Matrix:
 [[1768  216    1    0    0]
 [ 283 2259   88   15    5]
 [   2  155  320   64    0]
 [   0    9   60  359   24]
 [   0    5    3   51  111]]

📄 Classification Report:
               precision    recall  f1-score   support

           0       0.86      0.89      0.88      1985
           1       0.85      0.85      0.85      2650
           2       0.68      0.59      0.63       541
           3       0.73      0.79      0.76       452
           4       0.79      0.65      0.72       170

    accuracy                           0.83      5798
   macro avg       0.78      0.76      0.77      5798
weighted avg       0.83      0.83      0.83      5798



In [17]:
# ✅ Imports
!pip install  imbalanced-learn
!pip install seaborn 
!pip install scikit-learn


import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from imblearn.over_sampling import SMOTE
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
import seaborn as sns
import matplotlib.pyplot as plt

# ✅ Load Dataset
df = pd.read_csv("C:/Users/Admin/Downloads/city_day.csv") # replace with your file name

# ✅ Handle Missing Values
df.fillna(df.mean(numeric_only=True), inplace=True)

# ✅ Label Encode Target
le = LabelEncoder()
df['AQI_Bucket'] = le.fit_transform(df['AQI_Bucket'])  # convert labels to 0–4

# ✅ Feature/Target Split
X = df.drop(['AQI_Bucket', 'Date'], axis=1, errors='ignore')  # drop Date if exists
y = df['AQI_Bucket']

# ✅ Feature Scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ✅ Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.25, random_state=42, stratify=y)

# ✅ Balance Classes using SMOTE
smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

# ✅ Define Models
lgbm = LGBMClassifier(n_estimators=200, learning_rate=0.05, max_depth=10, random_state=42)
xgb = XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=10, use_label_encoder=False, eval_metric='mlogloss', random_state=42)
rf = RandomForestClassifier(n_estimators=200, max_depth=15, random_state=42)

# ✅ Ensemble Model
voting_clf = VotingClassifier(estimators=[
    ('lgbm', lgbm),
    ('xgb', xgb),
    ('rf', rf)
], voting='soft')

# ✅ Fit Models
voting_clf.fit(X_train_bal, y_train_bal)

# ✅ Predict
y_pred = voting_clf.predict(X_test)

# ✅ Evaluation
acc = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)
cr = classification_report(y_test, y_pred, target_names=le.classes_)

print(f"✅ Ensemble Accuracy: {acc:.4f}\n")
print("📊 Confusion Matrix:\n", cm)
print("\n📄 Classification Report:\n", cr)

# ✅ Confusion Matrix Heatmap
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=le.classes_, yticklabels=le.classes_)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()



[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


ValueError: could not convert string to float: 'Ahmedabad'

In [18]:
# ✅ Imports
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from imblearn.over_sampling import SMOTE
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
import seaborn as sns
import matplotlib.pyplot as plt

# ✅ Load Dataset
#df = pd.read_csv('air_quality_data.csv')  # replace with your dataset
df = pd.read_csv("C:/Users/Admin/Downloads/city_day.csv")
# ✅ Handle Missing Values
df.fillna(df.mean(numeric_only=True), inplace=True)

# ✅ Label Encode Target
le_target = LabelEncoder()
df['AQI_Bucket'] = le_target.fit_transform(df['AQI_Bucket'])  # convert labels to 0–4

# ✅ Encode All Categorical Features
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
categorical_cols = [col for col in categorical_cols if col != 'Date']  # drop Date if present

le_dict = {}
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    le_dict[col] = le  # store encoders if needed later

# ✅ Drop 'Date' column if exists
df = df.drop(['Date'], axis=1, errors='ignore')

# ✅ Feature/Target Split
X = df.drop('AQI_Bucket', axis=1)
y = df['AQI_Bucket']

# ✅ Feature Scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ✅ Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.25, random_state=42, stratify=y
)

# ✅ Balance Classes using SMOTE
smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

# ✅ Define Models
lgbm = LGBMClassifier(n_estimators=200, learning_rate=0.05, max_depth=10, random_state=42)
xgb = XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=10, use_label_encoder=False, eval_metric='mlogloss', random_state=42)
rf = RandomForestClassifier(n_estimators=200, max_depth=15, random_state=42)

# ✅ Ensemble Voting Classifier
voting_clf = VotingClassifier(estimators=[
    ('lgbm', lgbm),
    ('xgb', xgb),
    ('rf', rf)
], voting='soft')

# ✅ Fit Model
voting_clf.fit(X_train_bal, y_train_bal)

# ✅ Predict
y_pred = voting_clf.predict(X_test)

# ✅ Evaluate
acc = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)
cr = classification_report(y_test, y_pred, target_names=le_target.classes_)

print(f"✅ Ensemble Accuracy: {acc:.4f}\n")
print("📊 Confusion Matrix:\n", cm)
print("\n📄 Classification Report:\n", cr)

# ✅ Confusion Matrix Heatmap
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=le_target.classes_, yticklabels=le_target.classes_)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()


C:\python 3.9\lib\site-packages\sklearn\base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.005921 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3570
[LightGBM] [Info] Number of data points in the train set: 46347, number of used features: 14
[LightGBM] [Info] Start training from score -1.945910
[LightGBM] [Info] Start training from score -1.945910
[LightGBM] [Info] Start training from score -1.945910
[LightGBM] [Info] Start training from score -1.945910
[LightGBM] [Info] Start training from score -1.945910
[LightGBM] [Info] Start training from score -1.945910
[LightGBM] [Info] Start training from score -1.945910
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further s

C:\python 3.9\lib\site-packages\xgboost\core.py:158: UserWarning: [22:18:58] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
C:\python 3.9\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


TypeError: object of type 'float' has no len()

In [19]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

import lightgbm as lgb
import xgboost as xgb
import seaborn as sns
import matplotlib.pyplot as plt

# Load your dataset
df = pd.read_csv("C:/Users/Admin/Downloads/city_day.csv")

# Drop unnecessary columns (if any)
if 'Date' in df.columns:
    df.drop('Date', axis=1, inplace=True)

# Handle missing values (numerical columns → mean)
df.fillna(df.mean(numeric_only=True), inplace=True)

# Encode only object-type columns (categorical)
le_dict = {}
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].fillna("Unknown")
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col])
        le_dict[col] = le

# Separate features and target
X = df.drop(['AQI_Bucket'], axis=1)
y = df['AQI_Bucket']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Helper function to evaluate models
def evaluate_model(name, model, X_test, y_test):
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f"\n🔍 {name} Accuracy: {acc}\n")
    print("📊 Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
    print("\n📄 Classification Report:\n", classification_report(y_test, y_pred))

# 1. SVM
svm_model = SVC()
svm_model.fit(X_train, y_train)
evaluate_model("SVM", svm_model, X_test, y_test)

# 2. Naive Bayes
nb_model = GaussianNB()
nb_model.fit(X_train, y_train)
evaluate_model("Naive Bayes", nb_model, X_test, y_test)

# 3. Decision Tree
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train, y_train)
evaluate_model("Decision Tree", dt_model, X_test, y_test)

# 4. Random Forest
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
evaluate_model("Random Forest", rf_model, X_test, y_test)

# 5. K-Nearest Neighbors
knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train, y_train)
evaluate_model("KNN", knn_model, X_test, y_test)

# 6. LightGBM
lgb_model = lgb.LGBMClassifier()
lgb_model.fit(X_train, y_train)
evaluate_model("LightGBM", lgb_model, X_test, y_test)

# 7. XGBoost
xgb_model = xgb.XGBClassifier(use_label_encoder=False, eval_metric='mlogloss')
xgb_model.fit(X_train, y_train)
evaluate_model("XGBoost", xgb_model, X_test, y_test)



🔍 SVM Accuracy: 0.9443033688843745

📊 Confusion Matrix:
 [[ 203    0    0   34    0    0    0]
 [   0 1636    1    3    0  176    0]
 [   0   21  522    0    0    0   10]
 [   1   16    0 1594    0    0    0]
 [   0    0    0    0  257    0   15]
 [   0   47    0    0    0  900    0]
 [   0    0    3    0    2    0  466]]

📄 Classification Report:
               precision    recall  f1-score   support

           0       1.00      0.86      0.92       237
           1       0.95      0.90      0.93      1816
           2       0.99      0.94      0.97       553
           3       0.98      0.99      0.98      1611
           4       0.99      0.94      0.97       272
           5       0.84      0.95      0.89       947
           6       0.95      0.99      0.97       471

    accuracy                           0.94      5907
   macro avg       0.96      0.94      0.95      5907
weighted avg       0.95      0.94      0.94      5907


🔍 Naive Bayes Accuracy: 0.8660910783815812

📊 Conf

C:\python 3.9\lib\site-packages\xgboost\core.py:158: UserWarning: [22:21:28] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)



🔍 XGBoost Accuracy: 0.9969527679024885

📊 Confusion Matrix:
 [[ 237    0    0    0    0    0    0]
 [   0 1814    2    0    0    0    0]
 [   0    6  545    0    0    0    2]
 [   0    0    0 1611    0    0    0]
 [   0    0    0    0  271    0    1]
 [   0    0    0    0    0  947    0]
 [   0    0    3    0    4    0  464]]

📄 Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00       237
           1       1.00      1.00      1.00      1816
           2       0.99      0.99      0.99       553
           3       1.00      1.00      1.00      1611
           4       0.99      1.00      0.99       272
           5       1.00      1.00      1.00       947
           6       0.99      0.99      0.99       471

    accuracy                           1.00      5907
   macro avg       1.00      1.00      1.00      5907
weighted avg       1.00      1.00      1.00      5907



In [21]:
import joblib
joblib.dump(lgb_model , 'model.pkl')  # Save your trained model


['model.pkl']

In [23]:
!pip install streamlit
import streamlit as st
import numpy as np
import joblib

# Load the trained model
model = joblib.load("model.pkl")

st.title("🌫️ Air Quality Prediction")
st.write("Enter input features to predict the air quality category.")

# Example feature names (replace with actual feature names from your dataset)
feature_names = ['feature1', 'feature2', 'feature3', 'feature4', 'feature5']  # ← Update this
inputs = []

for name in feature_names:
    val = st.number_input(f"Enter {name}", format="%.5f")
    inputs.append(val)

if st.button("Predict"):
    prediction = model.predict([np.array(inputs)])
    st.success(f"Predicted Air Quality Class: **{int(prediction[0])}**")



[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


   ---------------------------------------- 0.0/10.1 MB ? eta -:--:--
   - -------------------------------------- 0.3/10.1 MB ? eta -:--:--
   -- ------------------------------------- 0.5/10.1 MB 1.9 MB/s eta 0:00:06
   ---- ----------------------------------- 1.0/10.1 MB 2.3 MB/s eta 0:00:04
   ------- -------------------------------- 1.8/10.1 MB 2.5 MB/s eta 0:00:04
   --------- ------------------------------ 2.4/10.1 MB 2.5 MB/s eta 0:00:04
   ------------ --------------------------- 3.1/10.1 MB 2.7 MB/s eta 0:00:03
   --------------- ------------------------ 3.9/10.1 MB 2.9 MB/s eta 0:00:03
   ------------------ --------------------- 4.7/10.1 MB 3.0 MB/s eta 0:00:02
   ---------------------- ----------------- 5.8/10.1 MB 3.1 MB/s eta 0:00:02
   --------------------------- ------------ 6.8/10.1 MB 3.3 MB/s eta 0:00:01
   ------------------------------- -------- 7.9/10.1 MB 3.5 MB/s eta 0:00:01
   ------------------------------------ --- 9.2/10.1 MB 3.7 MB/s eta 0:00:01
   ----------

2025-07-02 22:29:24.979 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-02 22:29:25.353 
  command:

    streamlit run C:\python 3.9\lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2025-07-02 22:29:25.354 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-02 22:29:25.355 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-02 22:29:25.356 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-02 22:29:25.357 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-02 22:29:25.359 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-02 22:29:25.361 Thread 'MainThread': missing Scrip